# Causal test of the clean candidate reserve

Lineage audit показал, что среди visible targets итоговый detection в 71.3% случаев даёт другой candidate, уже существовавший в clean target set. Здесь проверяется причинная гипотеза: **target скрывается, когда патч исчерпывает classification-score reserve этого фиксированного candidate ensemble**.

## Интервенции

- **Repair patched:** geometry и все остальные outputs остаются patched; только person logits выбранных clean target cells возвращаются к clean. Это тест необходимости подавления reserve.
- **Transplant clean:** geometry и все остальные outputs остаются clean; только person logits выбранных cells заменяются на реальные patched logits. Это тест достаточности подавления reserve.
- **Dose:** tracked, top-1/2/4/6/8/9/10/12 и весь clean target set.
- **Random controls:** столько же non-target cells, подобранных по feature level и ближайшему clean score.

Никакие оптимизированные attack directions не используются — только фактические logit changes исходного патча.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style='whitegrid',context='notebook')
REPO_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='CandidateRoutingAndAttackPath' else Path.cwd().resolve()
OUTPUT_ROOT=REPO_ROOT/'CandidateRoutingAndAttackPath/followup_outputs'
print(REPO_ROOT)

## Загрузка результатов

Тяжёлый model run выполняется отдельно на MPS с `tqdm`:

```bash
conda run -n IAD python -m CandidateRoutingAndAttackPath.run_followup_experiments reserve --device mps
```

Notebook только читает последний завершённый run и строит графики почти мгновенно.

In [ ]:
runs=sorted((p.parent for p in OUTPUT_ROOT.glob('candidate_reserve_*/summary.json')),key=lambda p:p.stat().st_mtime)
if not runs: raise FileNotFoundError('Сначала запустите reserve experiment командой из предыдущей клетки')
run_dir=runs[-1]
summary_json=json.loads((run_dir/'summary.json').read_text())
rows=pd.read_csv(run_dir/'candidate_reserve_rows.csv')
summary=pd.read_csv(run_dir/'candidate_reserve_summary.csv')
print(run_dir)
display(summary_json)
print(f"rows={len(rows):,}; examples={rows.example_id.nunique()}; conditions={rows.condition.nunique()}")

## 1. Endpoint sanity check

При `none` repair должен воспроизводить patched endpoint, а transplant — clean endpoint. Hidden-группы могут немного отличаться от старых labels из-за уже обнаруженного pipeline mismatch; дальнейшие rates считаются относительно baseline того же intervention pipeline.

In [ ]:
base=summary[summary.selection_kind=='none'].copy()
fig,ax=plt.subplots(figsize=(11,5))
sns.barplot(data=base,x='analysis_group',y='target_hidden_rate',hue='direction',ax=ax)
ax.set(ylim=(0,1),xlabel='',ylabel='target-hiding rate',title='Unmodified endpoints in the intervention pipeline')
ax.tick_params(axis='x',rotation=20)
plt.tight_layout(); plt.show()
display(base[['analysis_group','direction','n','target_hidden_rate','nms_only_hidden_rate']].round(3))

## 2. Necessity: repair the patched target reserve

Результат target-specific: matched random repair не восстанавливает ни одного hidden target при любом бюджете. Для low-confidence возвращение clean logit только tracked/top-1 cell уже даёт 90% recovery; top-2/4 и весь set повышают его до 93%/96%/99%. Для no-IoU рост более постепенный: 47.3% на tracked, 52.7% на top-4, 60.2% на top-8 и 61.3% на всём set. Следовательно, clean classification evidence выбранных target cells причинно необходимо почти для всех low-confidence failures и примерно для 60% no-IoU failures.

In [ ]:
DOSE=['0','top1','top2','top4','top6','top8','top9','top10','top12','all']
hidden_groups=['hidden_low_conf_match','hidden_no_iou_match']
repair=summary[(summary.direction=='repair_patched') & summary.analysis_group.isin(hidden_groups)].copy()
base_repair=repair[repair.selection_kind=='none'].copy()
expanded=[]
for kind in ['target','random']:
    q=base_repair.copy(); q['selection_kind']=kind; expanded.append(q)
repair_plot=pd.concat([repair[repair.selection_kind.isin(['target','random'])],*expanded],ignore_index=True)
repair_plot['dose']=repair_plot.budget_label.replace({'0':'0'}).astype(str)
repair_plot['dose']=pd.Categorical(repair_plot.dose,categories=DOSE,ordered=True)
fig,axes=plt.subplots(1,2,figsize=(15,5),sharey=True)
for ax,group in zip(axes,hidden_groups):
    q=repair_plot[(repair_plot.analysis_group==group)&repair_plot.dose.notna()].sort_values('dose')
    sns.lineplot(data=q,x='dose',y='recovery_rate',hue='selection_kind',marker='o',ax=ax)
    ax.set(ylim=(0,1),xlabel='restored cells',ylabel='recovery rate',title=group)
plt.suptitle('Necessity: target-specific repair versus matched random controls',y=1.03,fontsize=15)
plt.tight_layout(); plt.show()

`tracked` и `top1` определены независимо, но в этой выборке совпали во всех 400 примерах. Поэтому их одинаковые recovery rates — sanity check, а не два независимых результата. Важный смысл tracked repair: восстановление одного сильного clean logit часто достаточно, чтобы снова создать detection, но это ещё не означает, что исходный патч воздействовал только на эту клетку.

In [ ]:
single=summary[(summary.direction=='repair_patched') & summary.analysis_group.isin(hidden_groups) & summary.condition.isin(['tracked','top1','random_tracked','random_top1'])]
fig,ax=plt.subplots(figsize=(11,5))
sns.barplot(data=single,x='analysis_group',y='recovery_rate',hue='condition',ax=ax)
ax.set(ylim=(0,1),xlabel='',ylabel='recovery rate',title='One-cell repair: tracked versus strongest clean-set candidate')
plt.tight_layout(); plt.show()

## 3. Sufficiency: transplant actual patched logits into clean

Здесь получен главный collective-reserve result. Перенос patched logits для tracked, top-2/4/6/8 не скрывает ни одного target. Top-9 даёт лишь 6% low-confidence и 5% no-IoU hiding, но top-10 скачком достигает 86% и 56%; весь set — 90% и 57%. Clean target set обычно содержит около 10 cells. Значит подавление даже восьми самых сильных candidates недостаточно: оставшиеся один-два clean candidates удерживают объект. Hiding появляется только при почти полном исчерпании reserve. На visible controls перенос всего set скрывает лишь 4%/1%, поэтому эффект специфичен для фактических logit changes успешных атак, а не для самой операции замены.

In [ ]:
transplant=summary[summary.direction=='transplant_clean'].copy()
base_transplant=transplant[transplant.selection_kind=='none'].copy()
expanded=[]
for kind in ['target','random']:
    q=base_transplant.copy(); q['selection_kind']=kind; expanded.append(q)
transplant_plot=pd.concat([transplant[transplant.selection_kind.isin(['target','random'])],*expanded],ignore_index=True)
transplant_plot['dose']=pd.Categorical(transplant_plot.budget_label.astype(str),categories=DOSE,ordered=True)
fig,axes=plt.subplots(2,2,figsize=(15,10),sharey=True)
for ax,group in zip(axes.flat,['visible_target_winner','visible_non_target_winner',*hidden_groups]):
    q=transplant_plot[(transplant_plot.analysis_group==group)&transplant_plot.dose.notna()].sort_values('dose')
    sns.lineplot(data=q,x='dose',y='reproduced_hiding_rate',hue='selection_kind',marker='o',ax=ax)
    ax.set(ylim=(0,1),xlabel='transplanted cells',ylabel='reproduced hiding rate',title=group)
plt.suptitle('Sufficiency of actual patch-induced target-set logit changes',y=1.01,fontsize=15)
plt.tight_layout(); plt.show()

## 4. Matched-control sanity

Target и random conditions заменяют практически одинаковое число cells; небольшая недостача random возникает только когда в сохранённых top-50 не хватает pool того же размера. При этом максимальный recovery и reproduced hiding у random controls равны 0% во всех группах и бюджетах. Значит результат нельзя объяснить неспецифическим редактированием большого числа logits.

In [ ]:
control=summary[summary.selection_kind.isin(['target','random']) & summary.budget_label.isin(['top1','top2','top4','top6','top8','top9','top10','top12','all'])]
fig,ax=plt.subplots(figsize=(10,5))
sns.barplot(data=control,x='budget_label',y='mean_actual_k',hue='selection_kind',order=['top1','top2','top4','top6','top8','top9','top10','top12','all'],ax=ax)
ax.set(xlabel='condition',ylabel='mean replaced cells',title='Intervention-size matching')
plt.tight_layout(); plt.show()

## 5. Per-example transition heatmap

Строки — hidden examples, столбцы — размер target repair. Переход 1→0 показывает минимальный бюджет, при котором конкретный target восстановился; немонотонность указывает на NMS/candidate interaction.

In [ ]:
heat=rows[(rows.direction=='repair_patched') & (rows.selection_kind.isin(['none','target'])) & rows.analysis_group.isin(hidden_groups) & rows.budget_label.astype(str).isin(DOSE)].copy()
heat['dose']=pd.Categorical(heat.budget_label.astype(str),categories=DOSE,ordered=True)
matrix=heat.pivot_table(index=['analysis_group','example_id'],columns='dose',values='target_hidden',aggfunc='first').reindex(columns=DOSE)
matrix=matrix.sort_values(DOSE,ascending=False)
fig,ax=plt.subplots(figsize=(10,12))
sns.heatmap(matrix,cmap=['#4daf4a','#e41a1c'],vmin=0,vmax=1,cbar_kws={'ticks':[0,1],'label':'target hidden'},ax=ax)
ax.set(title='Example-level repair transitions',xlabel='restored target-set cells',ylabel='analysis group / example')
plt.show()

## 6. Четыре главных числа

In [ ]:
all_target=summary[(summary.selection_kind=='target') & (summary.budget_label=='all')].set_index(['direction','analysis_group'])
cards=[]
for group in hidden_groups:
    cards.append((f'repair all\n{group}',float(all_target.loc[('repair_patched',group),'recovery_rate']),'#4c78a8'))
for group in hidden_groups:
    cards.append((f'transplant all\n{group}',float(all_target.loc[('transplant_clean',group),'reproduced_hiding_rate']),'#e45756'))
fig,axes=plt.subplots(1,4,figsize=(17,3))
for ax,(label,value,color) in zip(axes,cards):
    ax.set_facecolor(color); ax.text(.5,.60,f'{value:.1%}',ha='center',va='center',fontsize=26,fontweight='bold',color='white')
    ax.text(.5,.20,label,ha='center',va='center',fontsize=10,color='white',wrap=True); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
display(all_target[['n','mean_actual_k','recovery_rate','reproduced_hiding_rate','mean_post_conf']].round(3))

## Критерий решения

Наблюдается target-specific dose response одновременно для necessity и sufficiency, при 0% эффекта matched random controls. Для low-confidence clean candidate reserve является почти полным causal mediator: all-set repair = 99%, transplant = 90%. Для no-IoU score reserve объясняет устойчивое ядро примерно 60%: repair = 61.3%, transplant = 57%; оставшаяся часть согласуется с уже установленным geometry/interaction компонентом и здесь повторно не исследуется. Самый важный результат — резкий sufficiency threshold между top-9 и top-10, показывающий не просто важность максимального candidate, а необходимость почти полного коллективного подавления примерно десяти ранее существовавших target cells.